# 01 · Exploration des données de rejeu

Ce notebook présente le jeu de **rejeu de référence** : 10 machines simulées pendant 8 h (seed 42), produit par
`python -m analyser.generator` à partir du simulateur réel (`simulator/`, non modifié), en temps accéléré.

- `batches.jsonl` : lots au format edge (`coffee/<device_id>/batch`), une lecture toutes les 2 s par machine ;
- `ground_truth.csv` : scénarios réellement injectés par le simulateur, attribués à chaque machine.

Le jeu est régénéré automatiquement s'il manque (≈ 1 s).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent  # AWS/Analyser
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))  # repli si iCloud masque le .venv (voir README)

import matplotlib.pyplot as plt
import pandas as pd

import _viz as viz
from analyser import generator
from analyser.config import Thresholds
from analyser.evaluation import analyse, episodes_from_ground_truth, evaluate, load_replay, sweep
from analyser.replay_files import read_ground_truth

viz.apply_style()
pd.set_option("display.precision", 2)

REPLAY = ROOT / "data" / "replay" / "ref"
if not (REPLAY / "batches.jsonl").exists():  # jeu de référence : 10 machines x 8 h, seed 42
    generator.write_replay(generator.generate(devices=10, hours=8, seed=42), REPLAY)

fetched = load_replay(REPLAY / "batches.jsonl")
ground_truth = read_ground_truth(REPLAY / "ground_truth.csv")
episodes = episodes_from_ground_truth(ground_truth, data_end_ms=fetched.now_ms)
eps = viz.episodes_frame(episodes)
print(f"{len(fetched.readings):,} lectures, {len(ground_truth):,} événements de vérité terrain, {len(episodes)} épisodes")

## 1. Vue d'ensemble

In [ ]:
df = viz.readings_frame(fetched.readings)
overview = df.groupby("device_id").agg(
    lectures=("timestamp", "size"),
    debut=("time", "min"),
    fin=("time", "max"),
    temp_max=("temperature_c", "max"),
    heap_min=("heap_free_bytes", "min"),
)
overview

Part du temps passée dans chaque statut (en %) :

In [ ]:
(pd.crosstab(df["device_id"], df["device_status"], normalize="index") * 100)[
    [s for s in viz.STATUS_ORDER if s in set(df["device_status"])]
].round(1)

## 2. Vérité terrain

Épisodes par machine et par scénario (un épisode = du début de la panne à son retour à la normale) :

In [ ]:
eps.pivot_table(index="device_id", columns="scenario", values="start", aggfunc="count", fill_value=0)

In [ ]:
eps.assign(duree_s=(eps["end"] - eps["start"]) / 1000).groupby("scenario")["duree_s"].describe()[["count", "min", "50%", "max"]]

## 3. Une machine sur 8 h

Chaque mesure a son propre panneau ; les zones colorées sont les épisodes courts de vérité terrain (la fuite mémoire, qui dure des heures, est visible directement sur le heap et détaillée en section 6).

In [ ]:
DEVICE = "coffee-machine-01"
dev = df[df["device_id"] == DEVICE]
viz.plot_device(dev, eps[eps["device_id"] == DEVICE], title=f"{DEVICE} — 8 h de télémétrie",
                shade=["thermal_runaway", "out_of_memory", "silent_device"]);

## 4. Zoom sur un emballement thermique

La température dépasse la consigne (94 °C) de 1,5 °C par lecture, jusqu'au seuil critique (112 °C) puis au redémarrage forcé.

In [ ]:
thermal = eps[(eps["device_id"] == DEVICE) & (eps["scenario"] == "thermal_runaway")].iloc[0]
window = (thermal.start_time - pd.Timedelta(minutes=2), thermal.end_time + pd.Timedelta(minutes=2))
viz.plot_device(dev, eps[eps["device_id"] == DEVICE], title="Emballement thermique", window=window,
                shade=["thermal_runaway"]);

## 5. Zoom sur une machine muette

Aucune lecture pendant 30 à 120 s : le trou dans les séries est le seul signal.

In [ ]:
silent = eps[eps["scenario"] == "silent_device"].iloc[0]
silent_dev = df[df["device_id"] == silent.device_id]
window = (silent.start_time - pd.Timedelta(minutes=3), silent.end_time + pd.Timedelta(minutes=3))
viz.plot_device(silent_dev, eps[eps["device_id"] == silent.device_id], title=f"Machine muette — {silent.device_id}", window=window, shade=["silent_device"]);

## 6. Fuite mémoire sur toute la flotte

Chaque mise à jour (`updating`) perd 8 000 octets de heap, jamais récupérés par le redémarrage normal. Après 6 fuites, le heap passe sous 5 000 octets : crash *out of memory* et remise à zéro.

In [ ]:
viz.plot_small_multiples(df, eps, "heap_free_bytes", "Heap libre par machine",
                         scenarios=["memory_leak_fota", "out_of_memory"]);

## Observations

- La température reste à ±0,3 °C de la consigne hors emballement : un seuil à consigne + 4 °C laisse une large marge.
- L'emballement atteint le seuil critique ≈ 24 s après son début : une alerte précoce (WARNING) doit tomber en quelques lectures.
- Le heap en `standby` est stable (±200 octets) et descend par marches de 8 000 octets après chaque mise à jour :
  comparer le heap en `standby` avant/après `updating` isole la fuite, alors que le `brewing` fait varier le heap de 2 000 octets.
- Les silences durent 30 à 120 s, bien moins que les 5 min du seuil « hors ligne » : ils sont détectés à part,
  à partir des trous entre horodatages de mesure.